In [ ]:
# %matplotlib widget

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from helper_functions import filtering, analysis, plotting, diagnostics
from obspy import read


In [ ]:
# ==================== Load data ====================
# dataType = 'sample'
# networkName = 'CAN'
# stationName = 'CAN'
# freqType = 'normal'

# raw_eventdata = pd.read_csv(
#     f'../data/{dataType}/{stationName}/{networkName}_measured_events_{freqType}_03102026.txt',
#     sep='|'
# )

# dataType = 'complete'
# networkName = 'AU'
# stationName = 'australia'
# freqType = 'normal'

# raw_eventdata = pd.read_csv(
#     f'../data/{dataType}/{stationName}/{networkName}_measured_events_{freqType}_03162026.txt',
#     sep='|'
# )

# dataType = 'complete'
# networkName = 'swp'
# stationName = 'swp'
# freqType = 'normal'

# raw_eventdata = pd.read_csv(
#     f'../data/{dataType}/{stationName}/{networkName}_measured_events_{freqType}_04032026.txt',
#     sep='|'
# )

dataType = 'complete'
networkName = 'AU'
stationName = 'australia'
freqType = 'normal'

raw_eventdata = pd.read_csv(
    f'../data/{dataType}/{stationName}/{networkName}_measured_events_{freqType}_04032026.txt',
    sep='|'
)

In [ ]:
# ==================== Prepare data ====================
# Calculate SI errors
eventData = filtering.calculate_si_errors(raw_eventdata)

epicentral_max=360
epicentral_min=0
depth_min=0
depth_max=1e5
snrbefore_min=2.0
snrafter_min=1.0
sta_lta_min=0
mag_max=7.0
ellipticity_min=1.0
ellipticity_max=1e4
si_error_max=0.7
incident_max=90
pierce_depth_min=0
delta_travel_max = 25


# Apply quality filters
eventData = filtering.filter_events(
    eventData,
    epicentral_max=epicentral_max,
    epicentral_min=epicentral_min,
    depth_min=depth_min,
    depth_max=depth_max,
    snrbefore_min=snrbefore_min,
    snrafter_min=snrafter_min,
    sta_lta_min=sta_lta_min,
    mag_max=mag_max,
    ellipticity_min=ellipticity_min,
    ellipticity_max=ellipticity_max,
    si_error_max=si_error_max,
    incident_max=incident_max,
    pierce_depth_min=pierce_depth_min,
    delta_travel_max=delta_travel_max
)

eventData = eventData.drop_duplicates(subset=['evtime','evstation'])

# eventData = eventData[(eventData['evphase']=='SKS')]
# eventData = eventData[eventData['wiggleSI_calc']<=2.6]

# Remove shallow events with large incident angles (keeps only events with large epicentral distances and smaller incident angles)
eventData = eventData[~((eventData['evdp'] < 400) & (eventData['evincident'] > 25))]

# # Remove deep events with very large incident angles  
eventData = eventData[~((eventData['evdp'] >= 400) & (eventData['evincident'] > 35))]

# ^^^^^^ these two filters are heavily used in the new analysis so keep them!!!

# eventData = eventData[eventData['evdp'] >= 400]


'''
april 5 notes
- added tilt to SI calculation and plotting (see analysis.py and spring_updates.ipynb)
- started looking at treating deep S phases and direct S phases separately (mainly dealing with incident angle and pierce depth filters)
- analysis of only deep S phases to identify trends and how this could be corraborated with sampling of anisotropy at tilted layer)
'''

print(f"Events after filtering: {len(eventData)}")

# previous codes with waveforms and initial plots

In [ ]:
depth = 200 
epic = 30

plt.figure(figsize=(10,8))
row_tick = 1

for i in range(len(eventData)):  #len(eventData)
    split = eventData.iloc[i]
    # print(split['waveform_path'])

#     wf = read(split['waveform_path'])
#     wf.detrend('demean')
#     wf.detrend('linear')
#     wf.filter("bandpass", freqmin=0.01, freqmax=0.125)

#     north = wf.select(component="N", channel='BH*')[0].data
#     east  = wf.select(component="E", channel='BH*')[0].data
#     sample_interval = wf[0].stats.delta


#     plt.subplot(4,len(eventData)//4,i+1)
#     plt.plot(north,c='b')
#     plt.plot(east,c='r')

# plt.show()


    if split['evphase'] == 'SKS':
        plt.plot(split['calcbaz'],split['wiggleSI_calc'],'o',c='b',alpha=0.3)
        plt.errorbar(split['calcbaz'],split['wiggleSI_calc'],yerr=split['SI_calc_error'],xerr=None,ecolor='blue',barsabove=True,alpha=0.5)

    if split['evphase'] == 'S':

        # if split['evdp'] >= depth:
        #     plt.plot(split['calcbaz'],split['wiggleSI_calc'],'o',c='g')
        #     plt.errorbar(split['calcbaz'],split['wiggleSI_calc'],yerr=split['SI_calc_error'],xerr=None,ecolor='green',barsabove=True,alpha=0.5)

        if split['evepic'] >= 60:
            plt.plot(split['calcbaz'],split['wiggleSI_calc'],'o',c='g',alpha=0.3)
            plt.errorbar(split['calcbaz'],split['wiggleSI_calc'],yerr=split['SI_calc_error'],xerr=None,ecolor='green',barsabove=True,alpha=0.5)

        else:
            plt.plot(split['calcbaz'],split['wiggleSI_calc'],'o',c='r',alpha=0.3)
            plt.errorbar(split['calcbaz'],split['wiggleSI_calc'],yerr=split['SI_calc_error'],xerr=None,ecolor='red',barsabove=True,alpha=0.5)




    
plt.xlabel('Polarization Azimuth')
plt.ylabel('Splitting Intensity')
plt.xticks(np.linspace(0,360,13))
plt.yticks(np.linspace(-3,3,13))

plt.show()

# ax1.errorbar(eventdata['calcbaz'].iloc[i],eventdata['wiggleSI_calc'].iloc[i],yerr=eventdata['SI_calc_error'].iloc[i],xerr=None,ecolor='blue',barsabove=True,alpha=0.5)

# plt.plot(eventData['calcbaz'],eventData['wiggleSI_calc'],'o')
# plt.errorbar(eventData['calcbaz'],eventData['wiggleSI_calc'],yerr=eventData['SI_calc_error'],xerr=None,ecolor='blue',barsabove=True,alpha=0.5)
# plt.show()

In [ ]:

# Define grid layout
n_plots = 12  # or use a smaller number for testing len(eventData)
ncols = 4  # number of columns
nrows = int(np.ceil(n_plots / ncols))  # calculate rows needed


# eventData = eventData[eventData['evstation'] == 'ARMA']
# # eventData = eventData[eventData['evphase'] == 'SKS']
# eventData = eventData[eventData['evbaz'] < 160]
# eventData = eventData[eventData['evbaz'] > 120]


eventData_shuffled = eventData.sample(frac=1).reset_index(drop=True)




# Create figure
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*4, nrows*3))
axes = axes.flatten()  # Convert to 1D array for easy indexing

for i in range(n_plots):
    split = eventData_shuffled.iloc[i]
    print(f"Processing {i+1}/{n_plots}: {split['waveform_path']}")
    
    try:
        wf = read(split['waveform_path'])
        wf.detrend('demean')
        wf.detrend('linear')
        wf.filter("bandpass", freqmin=0.01, freqmax=0.125)

        north = wf.select(component="N", channel='BH*')[0].data
        east = wf.select(component="E", channel='BH*')[0].data
        sample_interval = wf[0].stats.delta
        time = np.arange(len(north)) * sample_interval

        
        # Plot on the i-th axis
        axes[i].plot(time,north, c='b', label='North', linewidth=0.5)
        axes[i].plot(time,east, c='r', label='East', linewidth=0.5)
        axes[i].axvline(split['windowstart'], color='black', linestyle='--',linewidth=1.0, alpha=0.8)
        axes[i].axvline(split['windowend'], color='black', linestyle='--',linewidth=1.0, alpha=0.8)

        axes[i].set_title(f"{split['evstation']} - {split['evphase']}", fontsize=8)
        axes[i].tick_params(labelsize=6)
        
        if i == 0:  # Only show legend on first plot
            axes[i].legend(fontsize=6)
            
    except Exception as e:
        print(f"Failed on {i}: {e}")
        axes[i].text(0.5, 0.5, 'Failed', ha='center', va='center')
        axes[i].set_title(f"Error: {i}", fontsize=8)

# Hide unused subplots
for j in range(n_plots, len(axes)):
    axes[j].axis('off')

fig.suptitle(f"Sample of Waveforms, snrbefore={snrbefore_min}, snrafter={snrafter_min}, ellipmin={ellipticity_min}, SIerrmax={si_error_max}, epicmin={epicentral_min}", 
             fontsize=16, y=0.995)

plt.tight_layout()

# filepath = "/work/gcl3/BR/nicolasv03/senior_thesis/measurements/spring_plots/waveform_sample_plots/" + f'waveforms_snrb{snrbefore_min}_snra{snrafter_min}_ellip{ellipticity_min}_SIerr{si_error_max}_epic{epicentral_min}.png'
# plt.savefig(filepath, dpi=150, bbox_inches='tight')

plt.show()




In [ ]:
eventData

In [ ]:
# # ==================== Analyze single station ====================
# station_code = 'CAN'

# result = analysis.analyze_station(
#     event_data=eventData,
#     station_code=station_code,
#     bin_width=5,
#     bin_minimum=2,
#     s_depth=400,
#     weights={'sks': 1, 's': 1, 's_depth': 1, 's_shallow': 1},
#     incident_angle=20,
#     use_tilt=False
# )

# if result is not None:
#     print(f"\n{station_code} Results:")
#     print(f"  φ (fast axis): {result['phi']:.2f}°")
#     print(f"  δt (delay time): {result['dt']:.3f} s")
#     if 'tilt' in result:
#         print(f"  Tilt: {result['tilt']:.2f}°")
#     print(f"  N measurements: {result['n_events']}")
    
#     # ==================== Plot ====================
#     # Individual measurements
#     # fig1, ax1 = plotting.plot_si_measurements(result, wolf_data=fullwolf)
#     fig1, ax1 = plotting.plot_si_measurements(result, wolf_data=None)

#     plt.show()
    
#     # Binned measurements
#     # fig2, ax2 = plotting.plot_binned_measurements(result, wolf_data=fullwolf)
#     fig2, ax2 = plotting.plot_binned_measurements(result, wolf_data=None)

#     plt.show()
    
#     # Map
#     # fig3, ax3 = plotting.plot_station_map(result,s_depth=400)
#     plt.show()
    
#     # Or: comprehensive plot
#     # fig = plotting.plot_full_analysis(result, wolf_data=fullwolf)
#     # plt.show()
# else:
#     print(f"No results for {station_code}")


# ==================== Analyze single station ====================
station_code = 'DAV'

# sks phases only 
result_sks = analysis.analyze_station(
    event_data=eventData[eventData['evphase']=='SKS'],
    station_code=station_code,
    bin_width=5,
    bin_minimum=2,
    s_depth=400,
    weights={'sks': 1, 's': 1, 's_depth': 1, 's_shallow': 1},
    incident_angle=20,
    use_tilt=False
)

result_sks_and_deep_s = analysis.analyze_station(
    event_data=eventData[(eventData['evphase'] == 'SKS') | ((eventData['evphase'] == 'S') & (eventData['evdp'] >= 400))],
    station_code=station_code,
    bin_width=5,
    bin_minimum=2,
    s_depth=400,
    weights={'sks': 1, 's': 1, 's_depth': 1, 's_shallow': 1},
    incident_angle=20,
    use_tilt=False
)

result_all = analysis.analyze_station(
    event_data=eventData,
    station_code=station_code,
    bin_width=5,
    bin_minimum=2,
    s_depth=400,
    weights={'sks': 1, 's': 1, 's_depth': 1, 's_shallow': 1},
    incident_angle=20,
    use_tilt=False
)

# individual measurements (sks only)
fig1, ax1 = plotting.plot_si_measurements(result_sks, wolf_data=None)
plt.show()
    
# binned measurements (sks only)
fig2, ax2 = plotting.plot_binned_measurements(result_sks, wolf_data=None)
plt.show()

# individual measurements (sks and deep s phases)
fig1, ax1 = plotting.plot_si_measurements(result_sks_and_deep_s, wolf_data=None)
plt.show()

# binned measurements (sks and deep s phases)
fig2, ax2 = plotting.plot_binned_measurements(result_sks_and_deep_s, wolf_data=None)
plt.show()

# individual measurements (all phases)
fig1, ax1 = plotting.plot_si_measurements(result_all, wolf_data=None)
plt.show()

# binned measurements (all phases)
fig2, ax2 = plotting.plot_binned_measurements(result_all, wolf_data=None)
plt.show()


In [ ]:
station_code = 'ARMA'

# Get station events
from helper_functions import filtering
station_events = filtering.filter_station_events(eventData, station_code)

# Bin measurements
from helper_functions import analysis
binned = analysis.bin_measurements(
    station_events,
    bin_width=5,
    bin_minimum=1,
    s_depth=400,
    weights={'sks': 1, 's': 1, 's_depth': 1, 's_shallow': 1}
)

# Check for problems
print(f"Binned data shape: {binned.shape}")
print("\nChecking for issues:")
print(f"  NaN in azimuth: {binned['azimuth'].isna().sum()}")
print(f"  NaN in mean: {binned['mean'].isna().sum()}")
print(f"  NaN in weight: {binned['weight'].isna().sum()}")
print(f"  Inf in weight: {np.isinf(binned['weight']).sum()}")
print(f"  Zero weights: {(binned['weight'] == 0).sum()}")
print(f"  Zero margin: {(binned['margin'] == 0).sum()}")

print("\nWeight statistics:")
print(binned[['azimuth', 'mean', 'weight', 'margin']].describe())

print("\nProblematic rows:")
print(binned[~np.isfinite(binned['weight']) | (binned['weight'] == 0)])

# Analysis of each station and if S phases are necessary

In [ ]:
# Select station
station_code = 'SNZO'
bin_width = 30
bin_minimum = 3

# Run analysis
results = analysis.analyze_station_comparison(
    eventData, 
    station_code,
    bin_width=bin_width,
    bin_minimum=bin_minimum,
    s_depth=400
)

# Plot
fig, stats = analysis.plot_station_comparison(results, station_code, s_depth=400)
# plt.savefig(f'/work/gcl3/BR/nicolasv03/senior_thesis/measurements/spring_plots/single_station_SI_plots/{station_code}_width_{bin_width}_min_{bin_minimum}.png', dpi=300, bbox_inches='tight')
plt.show()

# Print summary
print(f"\n{'='*70}")
print(f"Summary Statistics for {station_code}")
print(f"{'='*70}")
for phase_key, label in [('sks', 'SKS Only'), ('sks_deep', 'SKS + Deep S'), ('all', 'All Phases')]:
    if phase_key in stats:
        s = stats[phase_key]
        print(f"\n{label}:")
        print(f"  N measurements: {s['n_measurements']}, N bins: {s['n_bins']}")
        print(f"  Binned:     φ={s['phi_binned']:6.1f}°, δt={s['dt_binned']:.3f}s")
        print(f"  Individual: φ={s['phi_indiv']:6.1f}°, δt={s['dt_indiv']:.3f}s")
        print(f"  Δφ = {s['delta_phi']:.1f}°, Δδt = {s['delta_dt']:.3f}s")

In [ ]:
plt.figure(figsize=(12,6))
plt.hist(eventData['snrbefore'], bins=np.linspace(0,20,201), alpha=0.5, label='SNR Before')
plt.xlim(0,20)
plt.xticks(np.linspace(0,20,21))
plt.xlabel('SNR')
plt.ylabel('Frequency')
plt.title('Distribution of SNR Before')
plt.legend()
plt.show()

plt.figure(figsize=(12,6))
plt.hist(eventData['snrafter'], bins=np.linspace(0,10,201), alpha=0.5, label='SNR After')
plt.xlim(0,10)
plt.xticks(np.linspace(0,10,21))
plt.xlabel('SNR')
plt.ylabel('Frequency')
plt.title('Distribution of SNR After')
plt.legend()
plt.show()

plt.figure(figsize=(12,6))
si_errors = eventData['SI_calc_error']
plt.hist(si_errors, bins=np.linspace(0,4,101), alpha=0.5, label='SI')

# plt.xlim(0,20)
# plt.xticks(np.linspace(0,20,21))
plt.xlabel('SI error')
plt.ylabel('Frequency')
plt.title('Distribution of SI error')
plt.legend()
plt.show()

plt.figure(figsize=(12,6))
si_errors = eventData['ellipticity']
plt.hist(si_errors, bins=np.linspace(0,20,21), alpha=0.5, label='Ellipticity')
plt.xlim(0,20)
plt.xticks(np.linspace(0,20,21))
plt.xlabel('Ellipticity')
plt.ylabel('Frequency')
plt.title('Distribution of Ellipticity')
plt.legend()
plt.show()

plt.figure(figsize=(12,6))
si_errors = eventData['sta_lta']
plt.hist(si_errors, bins=20, alpha=0.5, label='STA/LTA')
# plt.xlim(0,20)
# plt.xticks(np.linspace(0,20,21))
plt.xlabel('STA/LTA')
plt.ylabel('Frequency')
plt.title('Distribution of STA/LTA')
plt.legend()
plt.show()



In [ ]:
eventData.columns

In [ ]:
# station_code = 'STKA'
station_events = eventData[eventData['evstation'] == station_code]

# Plot 1: S phases colored by depth
fig1, ax1 = plotting.plot_s_phases_by_depth(station_events, station_code, s_depth=400)
plt.show()

# Plot 2: S phases colored by back-azimuth
fig2, ax2 = plotting.plot_s_phases_by_backazimuth(station_events, station_code, s_depth=400)
plt.show()

# Plot 3: Deep S only, colored by depth
fig3, ax3 = plotting.plot_s_phases_deep_only(station_events, station_code, 
                                             s_depth=400, color_by='depth')
plt.show()

# Plot 4: Deep S only, colored by back-azimuth
fig4, ax4 = plotting.plot_s_phases_deep_only(station_events, station_code,
                                             s_depth=400, color_by='backazimuth')
plt.show()

In [ ]:
# Two-panel: depth and back-azimuth coloring
fig5, axes = plotting.plot_s_phases_dual_view(station_events, station_code, s_depth=400)
plt.show()

# Source distribution analysis
fig6, axes = plotting.plot_polarization_vs_backazimuth_offset(station_events, station_code)
plt.show()

In [ ]:
# station_code = 'STKA'
station_events = eventData[eventData['evstation'] == station_code]

# Plot 1: S phases by incidence angle
fig1, ax1 = plotting.plot_s_phases_by_incidence(station_events, station_code)
plt.show()

# Plot 2: S phases by pierce depth
fig2, ax2 = plotting.plot_s_phases_by_pierce_depth(station_events, station_code)
plt.show()

# Plot 3: Comprehensive 6-panel comparison
fig3, axes = plotting.plot_s_phase_parameter_comparison(station_events, station_code)
plt.show()

# Plot 4: Incidence and pierce depth focused analysis
fig4, axes = plotting.plot_incidence_pierce_depth_grid(station_events, station_code)
plt.show()

In [ ]:
print(eventData['evstation'].unique())


In [ ]:


# Select station
station_code = 'ARMA'
station_events = eventData[eventData['evstation'] == station_code]
# station_events = station_events[station_events['evphase'] == 'S'] 
bin_width = 30
bin_minimum = 3

output_path = f'/work/gcl3/BR/nicolasv03/senior_thesis/measurements/spring_plots/single_station/{station_code}'

# # ========== Plot 1: SI Curves for Different Phase Types ==========
results = analysis.analyze_station_comparison(station_events, station_code, bin_width=bin_width, bin_minimum=bin_minimum, s_depth=400, use_tilt=True)
fig1, stats = analysis.plot_station_comparison(results, station_code, s_depth=400)
# plt.savefig(output_path+f'_si_curve_width_{bin_width}_min_{bin_minimum}.png', dpi=300, bbox_inches='tight')
plt.show()

# ========== Plot 2: S Phase Depths and Back-Azimuths ==========
fig2, axes = plotting.plot_s_phases_dual_view(station_events, station_code, s_depth=400)
# plt.savefig(output_path+f'_sPhase_depth_and_back_azimuth.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot 3: Comprehensive 6-panel comparison
fig3, axes = plotting.plot_s_phase_parameter_comparison(station_events, station_code)
plt.show()

# Plot 4: Incidence and pierce depth focused analysis
fig4, axes = plotting.plot_incidence_pierce_depth_grid(station_events, station_code)
plt.show()


In [ ]:
# Parameters
bin_width = 30
bin_minimum = 3
s_depth = 400

# Get all unique stations
station_list = eventData['evstation'].unique()
print(f"Found {len(station_list)} unique stations")
print(f"Stations: {sorted(station_list)}")

# Filter to stations with minimum data
station_counts = eventData.groupby('evstation').size()
station_list = station_counts[station_counts >= 10].index.tolist()  # At least 10 measurements
print(f"\nProcessing {len(station_list)} stations with ≥10 measurements")

# Loop through stations
for station_code in station_list:
    print(f"\n{'='*70}")
    print(f"Processing {station_code}...")
    print('='*70)
    
    try:
        station_events = eventData[eventData['evstation'] == station_code]
        output_path = f'/work/gcl3/BR/nicolasv03/senior_thesis/measurements/spring_plots/single_station/{station_code}'
        
        # ========== Plot 1: SI Curves for Different Phase Types ==========
        print(f"  Generating SI curve plot...")
        results = analysis.analyze_station_comparison(
            eventData, 
            station_code, 
            bin_width=bin_width, 
            bin_minimum=bin_minimum, 
            s_depth=s_depth
        )
        
        fig1, stats = analysis.plot_station_comparison(results, station_code, s_depth=s_depth)
        plt.savefig(output_path + f'_si_curve_width_{bin_width}_min_{bin_minimum}.png', 
                   dpi=300, bbox_inches='tight')
        plt.close()  # Close instead of show to save memory
        print(f"    ✓ Saved SI curve plot")
        
        # ========== Plot 2: S Phase Depths and Back-Azimuths ==========
        print(f"  Generating S-phase depth/baz plot...")
        fig2, axes = plotting.plot_s_phases_dual_view(station_events, station_code, s_depth=s_depth)
        
        if fig2 is not None:  # Check if S-phase data exists
            plt.savefig(output_path + f'_sPhase_depth_and_back_azimuth.png', 
                       dpi=300, bbox_inches='tight')
            plt.close()
            print(f"    ✓ Saved S-phase plot")
        else:
            print(f"    - No S-phase data for this station")
        
        print(f"  ✓ Completed {station_code}")
        
    except Exception as e:
        print(f"  ✗ Failed on {station_code}: {e}")
        plt.close('all')  # Clean up any open figures
        continue

print(f"\n{'='*70}")
print(f"✓ Processing complete for {len(station_list)} stations")
print('='*70)